In [1]:
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


# Baseline Graph

In [2]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── ICML style config ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{times}",
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.linewidth": 0.4,
    "grid.alpha": 0.5,
})

# ── Data ───────────────────────────────────────────────────────────────────────
attacks = ["Dija +\nHarmBench", "Context Nesting +\nHarmBench", "Vanilla\nHarmBench"]
asr_e   = [77.87, 75.74,  3.40]
asr_k   = [97.45, 88.51,  6.81]

x      = np.arange(len(attacks))
width  = 0.32

# ── Colors (colorblind-friendly) ───────────────────────────────────────────────
col_e = "#C04828"   # muted red
col_k = "#3B6EA5"   # muted blue

# ── Figure size: ICML single-column = 3.25 in; double = 6.75 in ───────────────
fig, ax = plt.subplots(figsize=(3.25, 2.6))

bars_e = ax.bar(x - width/2, asr_e, width, color=col_e, edgecolor="white",
                linewidth=0.3, zorder=3, label=r"$\mathrm{ASR}_e$")
bars_k = ax.bar(x + width/2, asr_k, width, color=col_k, edgecolor="white",
                linewidth=0.3, zorder=3, label=r"$\mathrm{ASR}_k$")

# Value labels on bars
for bar, val in zip(list(bars_e) + list(bars_k),
                    asr_e + asr_k):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1.2,
            f"{val:.1f}",
            ha="center", va="bottom",
            fontsize=6.5, color="#222222")

ax.set_xticks(x)
ax.set_xticklabels(attacks, linespacing=1.3)
ax.set_ylabel("Attack Success Rate (\%)")
ax.set_ylim(0, 112)
ax.yaxis.grid(True, zorder=0)
ax.set_axisbelow(True)

# Legend inside, upper right, away from tall bars
ax.legend(loc="upper right", frameon=False,
          handlelength=1.2, handletextpad=0.4, borderpad=0)

fig.tight_layout()
fig.savefig("baseline_asr.pdf")
fig.savefig("baseline_asr.png")
print("Saved: baseline_asr.pdf  /  baseline_asr.png")

<>:67: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
<>:67: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
/var/folders/wz/9w3ncb8s74s4kx1sxc1gxjtm0000gn/T/ipykernel_54156/3832212399.py:67: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
  ax.set_ylabel("Attack Success Rate (\%)")


Saved: baseline_asr.pdf  /  baseline_asr.png


# Context nesting remasking strategies 

In [3]:
"""
Generate ICML-style horizontal bar chart comparing remasking strategies
against jailbreak attacks (ASR_e metric, lower is better).
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── ICML style config (consistent with baseline figure) ───────────────────────
plt.rcParams.update({
    "text.usetex": True,              # set False if LaTeX not installed
    "text.latex.preamble": r"\usepackage{times}",
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.0,
    "xtick.major.size": 3,
    "ytick.major.size": 0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "grid.linewidth": 0.4,
    "grid.alpha": 0.5,
})

# ── Data ───────────────────────────────────────────────────────────────────────
strategies = [
    "Greedy (Default)",
    "Stochastic Annealing",
    "Random",
    "Adaptive Exponential",
    "SPD (Hybrid AR/MD)",
]
asr_values = [75.74, 70.21, 62.98, 52.34, 48.51]
is_ours    = [False, False, False, True, True]

# ── Colors (colorblind-friendly, consistent with baseline figure) ──────────────
COLOR_OURS     = "#C04828"   # muted red — our methods
COLOR_BASELINE = "#3B6EA5"   # muted blue — baselines

bar_colors = [COLOR_OURS if ours else COLOR_BASELINE for ours in is_ours]

# ── Figure size: ICML single-column = 3.25 in ─────────────────────────────────
fig, ax = plt.subplots(figsize=(3.25, 2.6))

y_pos = np.arange(len(strategies))

bars = ax.barh(
    y_pos,
    asr_values,
    height=0.58,
    color=bar_colors,
    edgecolor="white",
    linewidth=0.3,
    zorder=3,
)

# ── Value labels ──────────────────────────────────────────────────────────────
for bar, val in zip(bars, asr_values):
    ax.text(
        val + 0.8,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}",
        va="center",
        ha="left",
        fontsize=6.5,
        color="#222222",
    )

# ── Axes ──────────────────────────────────────────────────────────────────────
ax.set_yticks(y_pos)
ax.set_yticklabels(strategies)
ax.set_xlabel(r"$\mathrm{ASR}_e$ (\%) $\downarrow$", labelpad=4)
ax.set_xlim(0, 88)
ax.invert_yaxis()

ax.set_axisbelow(True)
ax.xaxis.grid(True, zorder=0)
ax.yaxis.grid(False)
ax.tick_params(axis="y", length=0)

# ── Legend ────────────────────────────────────────────────────────────────────
legend_ours = mpatches.Patch(facecolor=COLOR_OURS, edgecolor="none", label="Ours")
legend_base = mpatches.Patch(facecolor=COLOR_BASELINE, edgecolor="none", label="Baseline")
ax.legend(
    handles=[legend_ours, legend_base],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
    ncol=2,
    frameon=False,
    handlelength=1.2,
    handletextpad=0.4,
    columnspacing=1.5,
)

# ── Save ──────────────────────────────────────────────────────────────────────
fig.tight_layout(pad=0.4)
fig.savefig("fig_remasking_strategies.pdf")
fig.savefig("fig_remasking_strategies.png")
print("Saved: fig_remasking_strategies.pdf / .png")

Saved: fig_remasking_strategies.pdf / .png


In [23]:
"""                                                                                                                             
Generate two ICML-style vertical bar charts comparing remasking strategies:                                                     
(a) Structure Adherence Rate (SAR %)  — higher is better                                                                           
(b) Perplexity                        — lower is better                                                                            
"""                                                                                                                                
                                                                                                                                    
import matplotlib                                                                                                                  
matplotlib.use("Agg")
import matplotlib.pyplot as plt                                                                                                    
import matplotlib.patches as mpatches
import numpy as np                                                                                                                 
                
plt.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{times}",
    "font.family": "serif",                                                                                                        
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 12,                                                                                                               
    "axes.titlesize": 12,
    "axes.labelsize": 12,                                                                                                          
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,                                                                                                         
    "legend.fontsize": 10,
    "figure.dpi": 300,
    "savefig.dpi": 300,                                                                                                            
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,                                                                                                    
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,                                                                                                      
    "xtick.major.size": 3,                                                                                                         
    "ytick.major.size": 3,                                                                                                         
    "xtick.direction": "in",                                                                                                       
    "ytick.direction": "in",                                                                                                       
    "axes.spines.top": False,
    "axes.spines.right": False,                                                                                                    
    "grid.linewidth": 0.4,
    "grid.alpha": 0.5,
})

strategies  = [                                                                                                                    
    "Greedy (Default)",
    "Stochastic Annealing",                                                                                                        
    "Random",                                                                                                                      
    "Adaptive Exponential",
    "SPD (Hybrid AR/MD)",                                                                                                          
]               
sar_values  = [96.00, 89.00, 82.00, 72.00, 92.00]
ppl_values  = [9.60,  10.77, 14.20, 11.37, 11.79]                                                                                  
is_ours     = [False, False, False, True,  True]
                                                                                                                                    
COLOR_OURS     = "#C04828"                                                                                                         
COLOR_BASELINE = "#3B6EA5"                                                                                                         
                                                                                                                                    
bar_colors = [COLOR_OURS if o else COLOR_BASELINE for o in is_ours]

x_pos = np.arange(len(strategies))                                                                                                 
width = 0.55
                                                                                                                                    
legend_ours = mpatches.Patch(facecolor=COLOR_OURS, edgecolor="none", label="Ours")
legend_base = mpatches.Patch(facecolor=COLOR_BASELINE, edgecolor="none", label="Baseline")
                                                                                                                                    
# ── Side-by-side at full page width ──────────────────────────────────────────                                                    
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(6.75, 3.2))                                                                        
                                                                                                                                    
# ── (a) SAR ───────────────────────────────────────────────────────────────────                                                   
bars = ax_a.bar(x_pos, sar_values, width=width, color=bar_colors,                                                                  
                edgecolor="white", linewidth=0.3, zorder=3)                                                                        
                                                                                                                                    
for bar, val in zip(bars, sar_values):                                                                                             
    ax_a.text(bar.get_x() + bar.get_width() / 2, val + 0.8,                                                                        
                f"{val:.0f}", ha="center", va="bottom",                                                                              
                fontsize=9, color="#222222")                                                                                         
                                                                                                                                    
ax_a.set_xticks(x_pos)                                                                                                             
ax_a.set_xticklabels(strategies, rotation=45, ha="right")                                                                          
ax_a.set_ylabel(r"SAR (\%) $\uparrow$", labelpad=4)
ax_a.set_ylim(0, 112)                                                                                                              
ax_a.set_title("(a) Structure Adherence Rate", pad=6)                                                                              
ax_a.yaxis.grid(True, zorder=0)                                                                                                    
ax_a.xaxis.grid(False)                                                                                                             
ax_a.set_axisbelow(True)
                                                                                                                                    
# ── (b) Perplexity ────────────────────────────────────────────────────────────                                                   
bars = ax_b.bar(x_pos, ppl_values, width=width, color=bar_colors,
                edgecolor="white", linewidth=0.3, zorder=3)                                                                        
                
for bar, val in zip(bars, ppl_values):                                                                                             
    ax_b.text(bar.get_x() + bar.get_width() / 2, val + 0.15,
                f"{val:.2f}", ha="center", va="bottom",                                                                              
                fontsize=9, color="#222222")
                                                                                                                                    
ax_b.set_xticks(x_pos)                                                                                                             
ax_b.set_xticklabels(strategies, rotation=45, ha="right")                                                                          
ax_b.set_ylabel(r"Perplexity $\downarrow$", labelpad=4)                                                                            
ax_b.set_ylim(0, 18)                                                                                                               
ax_b.set_title("(b) Perplexity", pad=6)
ax_b.yaxis.grid(True, zorder=0)                                                                                                    
ax_b.xaxis.grid(False)
ax_b.set_axisbelow(True)                                                                                                           

# ── Shared legend ─────────────────────────────────────────────────────────────                                                   
fig.legend(     
    handles=[legend_ours, legend_base],                                                                                            
    loc="lower center",
    bbox_to_anchor=(0.5, 0.0),
    ncol=2, frameon=False,
    handlelength=1.2, handletextpad=0.4, columnspacing=2.0,
)                                                                                                                                  

fig.tight_layout(pad=0.4)                                                                                                          
fig.subplots_adjust(bottom=0.42, wspace=0.35)
fig.savefig("fig_remasking_sar.pdf")                                                                                               
fig.savefig("fig_remasking_sar.png")
print("Saved: fig_remasking_sar.pdf / .png")   

Saved: fig_remasking_sar.pdf / .png


# Perplexity and Structural adherence of harmless prompts

In [29]:
"""      
Generate two ICML-style vertical bar charts comparing remasking strategies:                                                     
(a) Structure Adherence Rate (SAR %)  — higher is better                                                                         
(b) Perplexity                        — lower is better                                                                          
"""                                                                                                                                
                                                                                                                                    
import matplotlib                                                                                                                  
matplotlib.use("Agg")                                                                                                              
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{times}",
    "font.family": "serif",                                                                                                        
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 11,                                                                                                               
    "axes.titlesize": 11,
    "axes.labelsize": 11,                                                                                                          
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,                                                                                                          
    "legend.fontsize": 9,
    "figure.dpi": 300,
    "savefig.dpi": 300,                                                                                                            
    "savefig.bbox": "tight",                                                                                                       
    "savefig.pad_inches": 0.02,                                                                                                    
    "axes.linewidth": 0.6,                                                                                                         
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,                                                                                                      
    "xtick.major.size": 3,                                                                                                         
    "ytick.major.size": 3,                                                                                                         
    "xtick.direction": "in",                                                                                                       
    "ytick.direction": "in",                                                                                                       
    "axes.spines.top": False,
    "axes.spines.right": False,                                                                                                    
    "grid.linewidth": 0.4,
    "grid.alpha": 0.5,
})

strategies  = [                                                                                                                    
    "Greedy (Default)",
    "Stochastic Annealing",                                                                                                        
    "Random",                                                                                                                      
    "Adaptive Exponential",                                                                                                        
    "SPD (Hybrid AR/MD)",                                                                                                          
]                                                                                                                                  
sar_values  = [96.00, 89.00, 82.00, 72.00, 92.00]
ppl_values  = [9.60,  10.77, 14.20, 11.37, 11.79]                                                                                  
is_ours     = [False, False, False, True,  True]
                                                                                                                                    
COLOR_OURS     = "#C04828"                                                                                                         
COLOR_BASELINE = "#3B6EA5"                                                                                                         
                                                                                                                                    
bar_colors = [COLOR_OURS if o else COLOR_BASELINE for o in is_ours]

x_pos = np.arange(len(strategies))                                                                                                 
width = 0.55
                                                                                                                                    
legend_ours = mpatches.Patch(facecolor=COLOR_OURS, edgecolor="none", label="Ours")
legend_base = mpatches.Patch(facecolor=COLOR_BASELINE, edgecolor="none", label="Baseline")
                                                                                                                                    
def add_legend(fig):
    fig.legend(                                                                                                                    
        handles=[legend_ours, legend_base],
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=2,                                                                                                                    
        frameon=False,                                                                                                             
        handlelength=1.2,                                                                                                          
        handletextpad=0.4,                                                                                                         
        columnspacing=1.5,
    )

# ── (a) Structure Adherence Rate ──────────────────────────────────────────────                                                   
fig, ax = plt.subplots(figsize=(3.25, 3.2))
                                                                                                                                    
bars = ax.bar(x_pos, sar_values, width=width, color=bar_colors,                                                                    
            edgecolor="white", linewidth=0.3, zorder=3)
                                                                                                                                    
for bar, val in zip(bars, sar_values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.8,
            f"{val:.0f}", ha="center", va="bottom",                                                                                
            fontsize=8, color="#222222")
                                                                                                                                    
ax.set_xticks(x_pos)                                                                                                               
ax.set_xticklabels(strategies, rotation=45, ha="right")
ax.set_ylabel(r"SAR (\%) $\uparrow$", labelpad=4)                                                                                  
ax.set_ylim(0, 112)
ax.set_title("(a) Structure Adherence Rate", pad=6)                                                                                
ax.yaxis.grid(True, zorder=0)
ax.xaxis.grid(False)                                                                                                               
ax.set_axisbelow(True)

add_legend(fig)                                                                                                                    
fig.tight_layout(pad=0.4)
fig.subplots_adjust(bottom=0.45)                                                                                                   
fig.savefig("fig_remasking_sar.pdf")
fig.savefig("fig_remasking_sar.png")                                                                                               
print("Saved: fig_remasking_sar.pdf / .png")                                                                                       
                                                                                                                                    
# ── (b) Perplexity ────────────────────────────────────────────────────────────                                                   
fig, ax = plt.subplots(figsize=(3.25, 3.2))                                                                                        
                                                                                                                                    
bars = ax.bar(x_pos, ppl_values, width=width, color=bar_colors,                                                                    
            edgecolor="white", linewidth=0.3, zorder=3)                                                                          
                                                                                                                                    
for bar, val in zip(bars, ppl_values):                                                                                             
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.15,                                                                         
            f"{val:.2f}", ha="center", va="bottom",                                                                                
            fontsize=8, color="#222222")                                                                                           
                                                                                                                                    
ax.set_xticks(x_pos)                                                                                                               
ax.set_xticklabels(strategies, rotation=45, ha="right")                                                                            
ax.set_ylabel(r"Perplexity $\downarrow$", labelpad=4)                                                                              
ax.set_ylim(0, 18)                                                                                                                 
ax.set_title("(b) Perplexity", pad=6)                                                                                              
ax.yaxis.grid(True, zorder=0)                                                                                                      
ax.xaxis.grid(False)                                                                                                               
ax.set_axisbelow(True)                                                                                                             
                
add_legend(fig)                                                                                                                    
fig.tight_layout(pad=0.4)
fig.subplots_adjust(bottom=0.45)
fig.savefig("fig_remasking_ppl.pdf")
fig.savefig("fig_remasking_ppl.png")                                                                                               
print("Saved: fig_remasking_ppl.pdf / .png")

Saved: fig_remasking_sar.pdf / .png
Saved: fig_remasking_ppl.pdf / .png


# Full Diffuguard SAR results

In [31]:
"""                                                                                                                                
(a) ICML-style vertical bar chart: Structure Adherence Rate (SAR %)                                                                
"""
                                                                                                                                    
import matplotlib                                                                                                                  
matplotlib.use("Agg")                                                                                                              
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches                                                                                              
import numpy as np                                                                                                                 
                                                                                                                                    
plt.rcParams.update({                                                                                                              
    "font.family": "serif",                                                                                                        
    "font.serif": ["Times New Roman", "DejaVu Serif"],                                                                             
    "font.size": 11,                                                                                                               
    "axes.titlesize": 11,                                                                                                          
    "axes.labelsize": 11,                                                                                                          
    "xtick.labelsize": 9,                                                                                                          
    "ytick.labelsize": 9,                                                                                                          
    "legend.fontsize": 9,                                                                                                          
    "figure.dpi": 300,                                                                                                             
    "savefig.dpi": 300,                                                                                                            
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,                                                                                                    
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,                                                                                                      
    "xtick.major.size": 3,
    "ytick.major.size": 3,                                                                                                         
    "xtick.direction": "in",
    "ytick.direction": "in",                                                                                                       
    "axes.spines.top": False,                                                                                                      
    "axes.spines.right": False,                                                                                                    
    "grid.linewidth": 0.4,                                                                                                         
    "grid.alpha": 0.5,
})                                                                                                                                 

strategies = [                                                                                                                     
    "Greedy (Default)",
    "Stochastic Annealing",
    "Fully Random",
    "Adaptive Exponential",
    "SPD (Hybrid AR/MD)",                                                                                                          
]
                                                                                                                                    
sar_values = [27.00, 27.00, 20.00, 22.00, 28.00]                                                                                   
is_ours    = [False, False, False, True,  True]
                                                                                                                                    
COLOR_OURS     = "#C04828"                                                                                                         
COLOR_BASELINE = "#3B6EA5"
bar_colors = [COLOR_OURS if o else COLOR_BASELINE for o in is_ours]                                                                
                
x_pos = np.arange(len(strategies))                                                                                                 
width = 0.55
                                                                                                                                    
legend_ours = mpatches.Patch(facecolor=COLOR_OURS, edgecolor="none", label="Ours")
legend_base = mpatches.Patch(facecolor=COLOR_BASELINE, edgecolor="none", label="Baseline")                                         
                                                                                                                                    
fig, ax = plt.subplots(figsize=(3.25, 3.2))
                                                                                                                                    
bars = ax.bar(x_pos, sar_values, width=width, color=bar_colors,
            edgecolor="white", linewidth=0.3, zorder=3)
                                                                                                                                    
for bar, val in zip(bars, sar_values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.4,                                                                          
            f"{val:.0f}", ha="center", va="bottom",                                                                                
            fontsize=8, color="#222222")                                                                                           
                                                                                                                                    
ax.set_xticks(x_pos)                                                                                                               
ax.set_xticklabels(strategies, rotation=45, ha="right")
ax.set_ylabel(r"SAR (\%) $\uparrow$", labelpad=4)                                                                                   
ax.set_ylim(0, 40)                                                                                                                 
ax.set_title("(a) Structure Adherence Rate", pad=6)                                                                                
ax.yaxis.grid(True, zorder=0)                                                                                                      
ax.xaxis.grid(False)                                                                                                               
ax.set_axisbelow(True)
                                                                                                                                    
fig.legend(                                                                                                                        
    handles=[legend_ours, legend_base],
    loc="lower center",                                                                                                            
    bbox_to_anchor=(0.5, 0.01),
    ncol=2, frameon=False,                                                                                                         
    handlelength=1.2, handletextpad=0.4, columnspacing=1.5,                                                                        
)                                                                                                                                  
                                                                                                                                    
fig.tight_layout(pad=0.4)                                                                                                          
fig.subplots_adjust(bottom=0.45)
fig.savefig("fig_remasking_sar_full_diff.pdf")                                                                                     
fig.savefig("fig_remasking_sar_full_diff.png")
print("Saved: fig_remasking_sar_full_diff.pdf / .png")                                                                             
                                                                                                                                    

"""                                                                                                                                
(b) ICML-style vertical bar chart: Perplexity (PPL) — lower is better
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt                                                                                                    
import matplotlib.patches as mpatches                                                                                              
import numpy as np                                                                                                                 
                                                                                                                                    
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 11,
    "axes.titlesize": 11,                                                                                                          
    "axes.labelsize": 11,
    "xtick.labelsize": 9,                                                                                                          
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 300,                                                                                                             
    "savefig.dpi": 300,
    "savefig.bbox": "tight",                                                                                                       
    "savefig.pad_inches": 0.02,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,                                                                                                      
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,                                                                                                         
    "ytick.major.size": 3,
    "xtick.direction": "in",                                                                                                       
    "ytick.direction": "in",                                                                                                       
    "axes.spines.top": False,
    "axes.spines.right": False,                                                                                                    
    "grid.linewidth": 0.4,
    "grid.alpha": 0.5,                                                                                                             
})                                                                                                                                 

strategies = [                                                                                                                     
    "Greedy (Default)",
    "Stochastic Annealing",                                                                                                        
    "Fully Random",
    "Adaptive Exponential",
    "SPD (Hybrid AR/MD)",
]

ppl_values = [33.06, 33.96, 52.06, 60.33, 31.98]                                                                                   
is_ours    = [False, False, False, True,  True]
                                                                                                                                    
COLOR_OURS     = "#C04828"
COLOR_BASELINE = "#3B6EA5"
bar_colors = [COLOR_OURS if o else COLOR_BASELINE for o in is_ours]                                                                
                                                                                                                                    
x_pos = np.arange(len(strategies))                                                                                                 
width = 0.55                                                                                                                       
                
legend_ours = mpatches.Patch(facecolor=COLOR_OURS, edgecolor="none", label="Ours")                                                 
legend_base = mpatches.Patch(facecolor=COLOR_BASELINE, edgecolor="none", label="Baseline")
                                                                                                                                    
fig, ax = plt.subplots(figsize=(3.25, 3.2))                                                                                        
                                                                                                                                    
bars = ax.bar(x_pos, ppl_values, width=width, color=bar_colors,                                                                    
            edgecolor="white", linewidth=0.3, zorder=3)
                                                                                                                                    
for bar, val in zip(bars, ppl_values):                                                                                             
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.6,                                                                          
            f"{val:.1f}", ha="center", va="bottom",                                                                                
            fontsize=8, color="#222222")                                                                                           

ax.set_xticks(x_pos)                                                                                                               
ax.set_xticklabels(strategies, rotation=45, ha="right")
ax.set_ylabel(r"Perplexity $\downarrow$", labelpad=4)
ax.set_ylim(0, 75)                                                                                                                 
ax.set_title("(b) Perplexity (PPL)", pad=6)                                                                                        
ax.yaxis.grid(True, zorder=0)                                                                                                      
ax.xaxis.grid(False)                                                                                                               
ax.set_axisbelow(True)                                                                                                             
                                                                                                                                    
fig.legend(                                                                                                                        
    handles=[legend_ours, legend_base],                                                                                            
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=2, frameon=False,
    handlelength=1.2, handletextpad=0.4, columnspacing=1.5,                                                                        
)
                                                                                                                                    
fig.tight_layout(pad=0.4)
fig.subplots_adjust(bottom=0.45)
fig.savefig("fig_remasking_ppl_full_diff.pdf")                                                                                     
fig.savefig("fig_remasking_ppl_full_diff.png")
print("Saved: fig_remasking_ppl_full_diff.pdf / .png")  

Saved: fig_remasking_sar_full_diff.pdf / .png
Saved: fig_remasking_ppl_full_diff.pdf / .png
